# 🧬 Self-Replicating Coding Agent — Colab Backend

Runs the full evolution pipeline locally on T4 GPU using **Qwen2.5-Coder** via Ollama.
No Groq/Gemini rate limits. Results are saved to Google Drive.

**Hardware:** T4 GPU (~2 Colab units/hr) — 100 units ≈ 50 hours of evolution

**Model:** `qwen2.5-coder:14b` (Q4, ~8 GB VRAM) — purpose-built for code generation

---

### Steps
1. **Cell 1** — Mount Drive + clone project  
2. **Cell 2** — Install Python deps  
3. **Cell 3** — Install Ollama + pull model (~5 GB, one-time)  
4. **Cell 4** — Start Ollama server  
5. **Cell 5** — Set API keys (optional fallback)  
6. **Cell 6** — Run evolution 🚀  
7. **Cell 7** — Show results  

In [ ]:
#@title 📂 Cell 1 — Mount Google Drive & set up project
#@markdown Results will be saved to Drive so they survive session disconnect.

from google.colab import drive
import os, shutil, subprocess

drive.mount('/content/drive')

# Project lives in Drive/MyDrive/SelfReplicatingAgent
DRIVE_DIR = '/content/drive/MyDrive/SelfReplicatingAgent'
WORK_DIR  = '/content/SelfReplicatingAgent'

os.makedirs(DRIVE_DIR, exist_ok=True)

# ── Option A: upload your project zip to Drive first, then unzip ─────────────
# If you already have the zip in Drive:
zip_path = f'{DRIVE_DIR}/2ndRunSelfReplicatingAgent.zip'
if os.path.exists(zip_path):
    print('📦 Found zip in Drive — extracting...')
    shutil.unpack_archive(zip_path, '/content/')
    # rename to standard work dir if needed
    extracted = [d for d in os.listdir('/content/') if 'SelfReplicating' in d and os.path.isdir(f'/content/{d}')]
    if extracted:
        os.rename(f'/content/{extracted[0]}', WORK_DIR)
    print(f'✅ Project extracted to {WORK_DIR}')

# ── Option B: clone from HuggingFace Spaces git ──────────────────────────────
elif not os.path.exists(WORK_DIR):
    print('🔗 Cloning from HuggingFace Spaces...')
    # Replace with your actual HF space repo URL
    HF_REPO = 'https://huggingface.co/spaces/Balaji33k/self-replicating-agent'
    subprocess.run(['git', 'clone', '--depth=1', HF_REPO, WORK_DIR], check=True)
    print(f'✅ Cloned to {WORK_DIR}')
else:
    print(f'✅ Project already at {WORK_DIR}')

# Symlink Drive/data ↔ local data so results persist across sessions
DATA_DRIVE = f'{DRIVE_DIR}/data'
DATA_LOCAL = f'{WORK_DIR}/data'
os.makedirs(DATA_DRIVE, exist_ok=True)
if os.path.exists(DATA_LOCAL) and not os.path.islink(DATA_LOCAL):
    # Merge any existing local data into Drive
    for f in os.listdir(DATA_LOCAL):
        src = f'{DATA_LOCAL}/{f}'
        dst = f'{DATA_DRIVE}/{f}'
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    shutil.rmtree(DATA_LOCAL)
if not os.path.islink(DATA_LOCAL):
    os.symlink(DATA_DRIVE, DATA_LOCAL)
    print(f'🔗 data/ → Drive ({DATA_DRIVE})')

# Similarly symlink generations output
GEN_DRIVE = f'{DRIVE_DIR}/generations'
GEN_LOCAL = f'{WORK_DIR}/generations'
os.makedirs(GEN_DRIVE, exist_ok=True)
# Keep gen_1 source intact — only symlink results subdirs
for gen in os.listdir(GEN_LOCAL) if os.path.exists(GEN_LOCAL) else []:
    results_local = f'{GEN_LOCAL}/{gen}/results'
    results_drive = f'{GEN_DRIVE}/{gen}/results'
    if os.path.isdir(results_local) and not os.path.islink(results_local):
        os.makedirs(results_drive, exist_ok=True)
        for f in os.listdir(results_local):
            shutil.copy2(f'{results_local}/{f}', f'{results_drive}/{f}')
        shutil.rmtree(results_local)
        os.symlink(results_drive, results_local)

print('✅ Drive persistence configured')
os.listdir(WORK_DIR)

In [ ]:
#@title 📦 Cell 2 — Install Python dependencies

import subprocess, sys
WORK_DIR = '/content/SelfReplicatingAgent'

print('Installing Python dependencies...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{WORK_DIR}/requirements.txt'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
else:
    print('✅ Dependencies installed')

# Verify key packages
pkgs = ['langchain_groq', 'langchain_google_genai', 'langchain_openai']
for pkg in pkgs:
    try:
        __import__(pkg)
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  ✗ {pkg} MISSING')

In [ ]:
#@title 🦙 Cell 3 — Install Ollama + pull Qwen2.5-Coder model
#@markdown This downloads ~8 GB once and caches to Drive so future sessions skip it.
#@markdown Takes ~3-5 minutes on first run.

import subprocess, os, shutil

DRIVE_DIR    = '/content/drive/MyDrive/SelfReplicatingAgent'
MODEL_CACHE  = f'{DRIVE_DIR}/ollama_models'
OLLAMA_MODEL = 'qwen2.5-coder:14b'  #@param ["qwen2.5-coder:14b", "qwen2.5-coder:7b", "qwen2.5:14b", "codellama:13b"]

# Install Ollama binary
if not shutil.which('ollama'):
    print('Installing Ollama...')
    subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, check=True
    )
    print('✅ Ollama installed')
else:
    print('✅ Ollama already installed')

# Use Drive as model store so we don't re-download each session
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['OLLAMA_MODELS'] = MODEL_CACHE
print(f'📁 Model cache: {MODEL_CACHE}')

# Check if model already in cache
model_dir = os.path.join(MODEL_CACHE, 'manifests', 'registry.ollama.ai', 'library',
                          OLLAMA_MODEL.replace(':', os.sep))
if os.path.exists(model_dir):
    print(f'✅ Model {OLLAMA_MODEL} already cached — skipping download')
else:
    print(f'⬇️  Pulling {OLLAMA_MODEL} (~8 GB for 14b, ~5 GB for 7b)...')
    # Start server briefly just for the pull
    srv = subprocess.Popen(
        ['ollama', 'serve'],
        env={**os.environ, 'OLLAMA_MODELS': MODEL_CACHE},
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    import time; time.sleep(3)  # wait for server to start
    result = subprocess.run(
        ['ollama', 'pull', OLLAMA_MODEL],
        env={**os.environ, 'OLLAMA_MODELS': MODEL_CACHE},
        capture_output=True, text=True
    )
    srv.terminate()
    if result.returncode != 0:
        print('Pull error:', result.stderr)
    else:
        print(f'✅ {OLLAMA_MODEL} downloaded and cached to Drive')

# Save the model name for later cells
with open('/content/ollama_model.txt', 'w') as f:
    f.write(OLLAMA_MODEL)

In [ ]:
#@title 🚀 Cell 4 — Start Ollama server

import subprocess, os, time, urllib.request

DRIVE_DIR   = '/content/drive/MyDrive/SelfReplicatingAgent'
MODEL_CACHE = f'{DRIVE_DIR}/ollama_models'

# Kill any existing Ollama
subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(1)

# Start fresh
env = {**os.environ, 'OLLAMA_MODELS': MODEL_CACHE}
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    env=env,
    stdout=open('/content/ollama.log', 'w'),
    stderr=subprocess.STDOUT
)

# Wait for it to be ready
print('Waiting for Ollama server...')
for i in range(20):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print(f'✅ Ollama server ready (took {i+1}s)')
        break
    except Exception:
        time.sleep(1)
        print(f'  ...waiting ({i+1}/20)')
else:
    print('❌ Ollama failed to start! Check /content/ollama.log')
    print(open('/content/ollama.log').read()[-2000:])

# Verify model is loaded
OLLAMA_MODEL = open('/content/ollama_model.txt').read().strip()
result = subprocess.run(['ollama', 'list'], capture_output=True, text=True,
                        env={**os.environ, 'OLLAMA_MODELS': MODEL_CACHE})
print('Available models:')
print(result.stdout)

# Set env var so llm_client.py picks it up
os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'
os.environ['OLLAMA_MODEL']    = OLLAMA_MODEL
print(f'🎯 OLLAMA_BASE_URL=http://localhost:11434  OLLAMA_MODEL={OLLAMA_MODEL}')

# Quick smoke test
import urllib.request, json
try:
    req = urllib.request.Request(
        'http://localhost:11434/api/generate',
        data=json.dumps({'model': OLLAMA_MODEL, 'prompt': 'Say OK', 'stream': False}).encode(),
        headers={'Content-Type': 'application/json'}
    )
    resp = json.loads(urllib.request.urlopen(req, timeout=60).read())
    print(f'🧪 Smoke test: "{resp["response"].strip()}"')
    print('✅ Ollama is working!')
except Exception as e:
    print(f'⚠️  Smoke test failed: {e}')
    print('   (If model is not loaded yet, re-run Cell 3)')

In [ ]:
#@title 🔑 Cell 5 — API keys (optional fallback if Ollama fails)
#@markdown Leave blank if you only want to use local Ollama.
#@markdown Keys are only used if Ollama is unreachable.

import os
from google.colab import userdata

# Try Colab Secrets first (Colab ▶ Secrets panel)
def get_secret(key):
    try:
        val = userdata.get(key)
        return val if val else ''
    except Exception:
        return ''

groq_key   = get_secret('GROQ_API_KEY')   or ''  #@param {type:"string"}
gemini_key = get_secret('GEMINI_API_KEY') or ''  #@param {type:"string"}

if groq_key:
    os.environ['GROQ_API_KEY'] = groq_key
    print('✅ GROQ_API_KEY set (fallback ready)')
if gemini_key:
    os.environ['GEMINI_API_KEY'] = gemini_key
    print('✅ GEMINI_API_KEY set (fallback ready)')

# Ollama is the primary — these are fallbacks only
if os.environ.get('OLLAMA_BASE_URL'):
    print('🦙 Primary provider: Ollama (local, no rate limits)')
elif groq_key:
    print('⚡ Primary provider: Groq (rate limits apply)')
elif gemini_key:
    print('✨ Primary provider: Gemini (rate limits apply)')
else:
    print('❌ WARNING: No provider configured!')

In [ ]:
#@title 🧬 Cell 6 — Run Evolution Pipeline
#@markdown Starts from Gen 1. Each generation runs 20 tasks, then spawns the next.
#@markdown Results saved to Drive continuously.

import subprocess, sys, os, time, json
from pathlib import Path

WORK_DIR     = '/content/SelfReplicatingAgent'
GEN1_DIR     = f'{WORK_DIR}/generations/gen_1'
DATA_DIR     = f'{WORK_DIR}/data'
MAX_GENS     = 5  #@param {type:"integer"}
START_GEN    = 1  #@param {type:"integer"}

os.makedirs(DATA_DIR, exist_ok=True)

# Clear any leftover stop flags
stop_flag = Path(f'{DATA_DIR}/stop.flag')
if stop_flag.exists():
    stop_flag.unlink()
    print('🗑️  Cleared old stop.flag')

print(f'🚀 Starting evolution from Gen {START_GEN}, max {MAX_GENS} generations')
print(f'🦙 LLM: {os.environ.get("OLLAMA_MODEL", "(API fallback)")} via {os.environ.get("OLLAMA_BASE_URL", "API")}')
print('=' * 60)

# Run main.py for the starting generation
gen_dir = Path(WORK_DIR) / 'generations' / f'gen_{START_GEN}'
if not gen_dir.exists():
    print(f'❌ Gen {START_GEN} directory not found: {gen_dir}')
    raise SystemExit(1)

log_file = open(f'{DATA_DIR}/colab_run.log', 'w', buffering=1)

proc = subprocess.Popen(
    [sys.executable, 'main.py', '--gen', str(START_GEN)],
    cwd=str(gen_dir),
    env={**os.environ},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

print(f'▶ Gen {START_GEN} PID={proc.pid}')

# Live output
try:
    for line in proc.stdout:
        line = line.rstrip()
        if line:
            print(line)
            log_file.write(line + '\n')
            log_file.flush()
except KeyboardInterrupt:
    print('\n⏹ Interrupted by user')
    proc.terminate()
finally:
    log_file.close()

ret = proc.wait()
print(f'\n✅ Gen {START_GEN} finished (exit code {ret})')

# Show quick result summary
results_dir = gen_dir / 'results'
if results_dir.exists():
    result_files = list(results_dir.glob('*.json'))
    print(f'\n📊 Results: {len(result_files)} tasks')
    passed = failed = 0
    for f in result_files:
        try:
            d = json.loads(f.read_text())
            if d.get('status') == 'success': passed += 1
            else: failed += 1
        except: pass
    print(f'   ✅ Passed: {passed}  ❌ Failed: {failed}  📈 Pass rate: {passed/(passed+failed)*100:.1f}%' if (passed+failed) > 0 else '   No results yet')

In [ ]:
#@title 📊 Cell 7 — View Results

import json, os
from pathlib import Path

WORK_DIR = '/content/SelfReplicatingAgent'

gens_dir = Path(WORK_DIR) / 'generations'
all_gens = sorted([d for d in gens_dir.iterdir() if d.is_dir() and d.name.startswith('gen_')])

print('=' * 60)
print('EVOLUTION RESULTS SUMMARY')
print('=' * 60)

for gen_dir in all_gens:
    results_dir = gen_dir / 'results'
    if not results_dir.exists():
        continue
    result_files = list(results_dir.glob('*.json'))
    passed = failed = errors = 0
    error_types = {}
    for f in result_files:
        try:
            d = json.loads(f.read_text())
            status = d.get('status', 'unknown')
            if status == 'success':
                passed += 1
            else:
                failed += 1
                err = d.get('error_type', d.get('error', 'unknown'))[:40]
                error_types[err] = error_types.get(err, 0) + 1
        except Exception as e:
            errors += 1

    total = passed + failed
    pct   = f'{passed/total*100:.1f}%' if total > 0 else 'N/A'
    bar   = '█' * passed + '░' * failed

    print(f'\n{gen_dir.name.upper()}  [{bar[:20]}] {pct} ({passed}/{total})')
    if error_types:
        for err, cnt in sorted(error_types.items(), key=lambda x: -x[1])[:5]:
            print(f'  • {err}: {cnt}x')

# Show model usage if available
usage_file = Path(WORK_DIR) / 'data' / 'token_usage.json'
if usage_file.exists():
    try:
        usage = json.loads(usage_file.read_text())
        print(f'\n📈 Total tokens used: {usage.get("total_tokens", 0):,}')
    except: pass

# Show Ollama log tail
log_file = Path('/content/ollama.log')
if log_file.exists():
    lines = log_file.read_text().splitlines()[-5:]
    print('\n🦙 Ollama log (last 5 lines):')
    for l in lines:
        print(f'  {l}')

In [ ]:
#@title ⏰ Cell 8 — Keep session alive (run in background)
#@markdown Prevents Colab from disconnecting due to idle timeout during long runs.
#@markdown Run this cell BEFORE Cell 6, then run Cell 6.

import threading, time, datetime

_keep_alive = True

def _heartbeat():
    while _keep_alive:
        print(f'💓 {datetime.datetime.now().strftime("%H:%M:%S")} — session alive', end='\r')
        time.sleep(30)

t = threading.Thread(target=_heartbeat, daemon=True)
t.start()
print('✅ Keep-alive thread started (prints heartbeat every 30s)')
print('   To stop: set _keep_alive = False in a new cell')